In [1]:
import pandas as pd
import numpy as np
from sklearn.cluster import KMeans

import torch
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import NMF
from sklearn.cluster import KMeans
from sklearn.metrics.pairwise import cosine_similarity

from transformers import AutoTokenizer, AutoModelForSequenceClassification

from sentence_transformers import SentenceTransformer
from collections import Counter

/Users/ariellerabinovich/anaconda3/envs/ds_new/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [13]:
bills = pd.read_csv("../data/bills_cleaned.csv")
bills.head()

,Legislation Number,URL,Congress,Title,Sponsor,Date of Introduction,Committees,Latest Action,Latest Action Date,Number of Cosponsors
0,H.R. 7933,https://www.congress.gov/bill/119th-congress/h...,119th Congress (2025-2026),"To amend title 10, United States Code, to expa...","Watson Coleman, Bonnie [Rep.-D-NJ-12]",2026-03-12,House - Armed Services,Referred to the House Committee on Armed Servi...,2026-03-12,0
1,H.R. 7932,https://www.congress.gov/bill/119th-congress/h...,119th Congress (2025-2026),HONOR Gold Star Families Act,"Van Epps, Matt [Rep.-R-TN-7]",2026-03-12,House - Armed Services,Referred to the House Committee on Armed Servi...,2026-03-12,11
2,H.R. 7931,https://www.congress.gov/bill/119th-congress/h...,119th Congress (2025-2026),To amend the Employee Retirement Income Securi...,"Van Drew, Jefferson [Rep.-R-NJ-2]",2026-03-12,House - Education and Workforce,Referred to the House Committee on Education a...,2026-03-12,1
3,H.R. 7930,https://www.congress.gov/bill/119th-congress/h...,119th Congress (2025-2026),"To amend title 18, United States Code, to incr...","Tlaib, Rashida [Rep.-D-MI-12]",2026-03-12,House - Judiciary,Referred to the House Committee on the Judiciary.,2026-03-12,6
4,H.R. 7929,https://www.congress.gov/bill/119th-congress/h...,119th Congress (2025-2026),To direct the Secretary of Transportation to i...,"Titus, Dina [Rep.-D-NV-1]",2026-03-12,"House - Science, Space, and Technology","Referred to the House Committee on Science, Sp...",2026-03-12,0


In [14]:
bills["category"] = ""

In [16]:
embedder = SentenceTransformer("all-MiniLM-L6-v2",)
title_embeddings = embedder.encode(bills["Title"].tolist(), show_progress_bar=True)

Batches: 100%|██████████| 79/79 [00:13<00:00,  5.89it/s]


In [17]:
#fit KMeans with n = 7
kmeans = KMeans(n_clusters=7, random_state=42, n_init=10)
bills["kmeans_cluster"] = kmeans.fit_predict(title_embeddings)

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
/Users/ariellerabinovich/anaconda3/envs/ds_new/lib/python3.11/site-packages/threadpoolctl.py:1226: RuntimeWarning: 
Found Intel OpenMP ('libiomp') and LLVM OpenMP ('libomp') loaded at
the same time. Both libraries are known to be incompatible and this
can cause random crashes or deadlocks on Linux when loaded in the
same Python program.
Using threadpoolctl may cause crashes or deadlocks. For more
information and possible workarounds, please see
    https://github.com/joblib/threadpoolctl/blob/master/multiple_openmp.md

  warnings.warn(msg, RuntimeWarning)


In [21]:
# find and print 5 sentences closest to the centroid of each cluster
print("Discovered topics (KMeans on embeddings):\n")
for c in range(7):
    cluster_mask = bills["kmeans_cluster"] == c
    cluster_embs = title_embeddings[cluster_mask]
    centroid = kmeans.cluster_centers_[c]
    dists = cosine_similarity([centroid], cluster_embs)[0]
    top_idx = dists.argsort()[-5:][::-1]
    cluster_reviews = bills[cluster_mask].iloc[top_idx]
    print(f"  ── Cluster {c} ({cluster_mask.sum()} titles) ──")
    for _, row in cluster_reviews.iterrows():
        print(f"    \"{row['Title'][:200]}...\"")
    print()

Discovered topics (KMeans on embeddings):

  ── Cluster 0 (330 titles) ──
    "___ Act..."
    "BASIC Act..."
    "REAL Act..."
    "PLAY Act..."
    "MOVE Act..."

  ── Cluster 1 (253 titles) ──
    "Improving Veteran Access to Care Act..."
    "Veteran Benefits Enhancement Act..."
    "CARING for Our Veterans Health Act of 2025..."
    "Connecting Veterans to Care Act of 2025..."
    "Veterans HOPE Act..."

  ── Cluster 2 (269 titles) ──
    "Farm and Family Relief Act..."
    "Rural Housing Regulatory Relief Act..."
    "Empowering Rural Communities Act..."
    "FARMS Act..."
    "Assistance for Rural Water Systems Act of 2026..."

  ── Cluster 3 (220 titles) ──
    "To provide for the transfer to the Office for State and Local Law Enforcement of the Department of Homeland Security of the National Threat Evaluation and Reporting Program of the Department, and for ..."
    "To reauthorize Trade Adjustment Assistance programs, and for other purposes...."
    "To provide the Secretary 

In [ ]:
# pick distinct categories

# veterans
# rural
# funding
# infrastructure

# some more i think would be worth being distinct

# education
# healthcare
# environment
# public safety
# housing

In [ ]:
topic_descriptions = {
    "veterans":"focused on the rights, benefits, healthcare, housing, or services for military veterans and their families.",
    "rural": "addressing the distinct needs of rural communities, including agriculture, broadband access, rural healthcare, and economic development in non-urban areas.",
    "funding":"",
    "infrastructure": "",
    "education":       "",
    "healthcare":      "",
    "environment":   "",
    "public_safety":      "",
    "housing":   "",
}